In [1]:
import sys
import os
import gc
from pathlib import Path

import pandas as pd

sys.path.append(os.path.abspath("../../../"))
PROJECT_ROOT = "../../../"

from preprocessing.Weibo2014.preprocessing import (
    load_weibo2014_data
)
from preprocessing.general.filtering import (
    apply_filters_to_dataset,
    bands
)
import preprocessing.general.feature_extraction as fe


# ============================================================
# Dataset metadata
# ============================================================

DATASET_NAME = "weibo2014"

# Weibo2014 contains one acquisition session.
SESSION_NAME = "session_01"


# ============================================================
# Channel configuration
# ============================================================

# Simple identifier for this electrode configuration.
ELECTRODE_SETUP = "setup_01"

# Electrode configuration used by setup_01.
selected_channels = [
    "C3",
    "Cz",
    "C4",
]

# Use all available Weibo2014 EEG channels instead:
# selected_channels = None


# ============================================================
# Paths
# ============================================================

root_weibo = os.path.join(
    PROJECT_ROOT,
    "Datasets/Weibo2014/original/Weibo2014/"
)

output_csv = Path(
    os.path.join(
        PROJECT_ROOT,
        "Datasets/Weibo2014/processed/",
        f"Weibo2014_features_{ELECTRODE_SETUP}.csv",
    )
)


# ============================================================
# Feature configuration
# ============================================================

extract_config = {
    "mean": {"function": fe.extract_mean},
    "std": {"function": fe.extract_std},
    "mom": {"function": fe.extract_moments},
    "min": {"function": fe.extract_min},
    "max": {"function": fe.extract_max},
    "cov": {"function": fe.extract_covariance},
    "eig": {"function": fe.extract_eigenvalues},

    # "logcov": {
    #     "function": fe.extract_logcov
    # },

    # "fft": {
    #     "function": fe.extract_fft,
    #     "params": {"ntop": 5},
    # },

    "h_diff": {"function": fe.extract_halves_diff},
    "q_stats": {"function": fe.extract_quarters_stats},
    "logvar": {"function": fe.extract_logvar},
}


# ============================================================
# Incremental processing configuration
# ============================================================

subjects = list(range(1, 11))

# With only 10 subjects this can reasonably be 5.
# Reduce to 1 for minimum memory usage.
SUBJECT_BATCH_SIZE = 5


# ============================================================
# Prepare output
# ============================================================

output_csv.parent.mkdir(
    parents=True,
    exist_ok=True,
)

# Prevent results from being appended to an old file.
if output_csv.exists():
    output_csv.unlink()

first_write = True
total_rows = 0


# ============================================================
# Load, filter, extract, validate, and save incrementally
# ============================================================

for start in range(
    0,
    len(subjects),
    SUBJECT_BATCH_SIZE,
):

    subject_batch = subjects[
        start:start + SUBJECT_BATCH_SIZE
    ]

    print(
        f"\nProcessing subjects "
        f"{subject_batch[0]}–{subject_batch[-1]}"
    )

    # --------------------------------------------------------
    # Load current batch
    # --------------------------------------------------------

    batch_data = load_weibo2014_data(
        root=root_weibo,
        config={
            "subjects": subject_batch,
            "channels": selected_channels,

            # Keep only the three classes shared across datasets.
            "classes": [
                "left_hand_imagery",
                "right_hand_imagery",
                "both_feet_imagery",
            ],
        },
    )

    if not batch_data:
        print("⚠️ No data loaded for this batch.")
        continue

    print("✅ Data loading complete.")

    # --------------------------------------------------------
    # Filtering and resampling
    # --------------------------------------------------------

    filtered_data = apply_filters_to_dataset(
        dataset=batch_data,
        config={
            "original_fs": 200,
        },
    )

    print("✅ Filtering complete.")

    # --------------------------------------------------------
    # Feature extraction
    # --------------------------------------------------------

    df_batch = fe.extract_features_to_dataframe(
        dataset=filtered_data,
        extract_config=extract_config,
        band_labels=bands,
        dataset_name=DATASET_NAME,
        session_name=SESSION_NAME,
    )

    if df_batch.empty:
        print("⚠️ No features generated for this batch.")

        del batch_data
        del filtered_data
        del df_batch

        gc.collect()
        continue

    print(
        f"✅ Feature extraction complete: "
        f"{df_batch.shape}"
    )

    # --------------------------------------------------------
    # Validate current batch
    # --------------------------------------------------------

    fe.validate_feature_dataframe(df_batch)

    print("✅ Batch validation complete.")

    # --------------------------------------------------------
    # Append batch to CSV
    # --------------------------------------------------------

    df_batch.to_csv(
        output_csv,
        mode="w" if first_write else "a",
        header=first_write,
        index=False,
    )

    if first_write:
        display(df_batch.head())
        first_write = False

    total_rows += len(df_batch)

    print(
        f"✅ Batch saved. "
        f"Total rows written: {total_rows}"
    )

    # --------------------------------------------------------
    # Release memory
    # --------------------------------------------------------

    del batch_data
    del filtered_data
    del df_batch

    gc.collect()


# ============================================================
# Final validation and report
# ============================================================

if first_write:
    print("⚠️ No feature data were written.")

else:
    print("\nValidating complete saved dataset...")

    df_complete = pd.read_csv(output_csv)

    fe.validate_feature_dataframe(df_complete)

    if len(df_complete) != total_rows:
        raise ValueError(
            "The number of rows in the saved CSV does not match "
            f"the number written: {len(df_complete)} versus "
            f"{total_rows}."
        )

    print("✅ Complete dataset validation passed.")
    print("✅ Complete feature extraction finished.")
    print(f"✅ Electrode setup: {ELECTRODE_SETUP}")
    print(f"✅ Channels: {selected_channels}")
    print(f"✅ Total rows: {total_rows}")
    print(f"✅ Saved to: {output_csv}")

    del df_complete
    gc.collect()


Processing subjects 1–5
✅ Data loading complete.
✅ Filtering complete.


Subjects: 100%|██████████| 5/5 [00:03<00:00,  1.37it/s]


✅ Feature extraction complete: (1200, 95)
✅ Batch validation complete.


,dataset,subject,session,trial_index,label,b8_12_mean_C3,b8_12_mean_Cz,b8_12_mean_C4,b8_12_std_C3,b8_12_std_Cz,...,b13_30_q_1_C4,b13_30_q_2_C3,b13_30_q_2_Cz,b13_30_q_2_C4,b13_30_q_3_C3,b13_30_q_3_Cz,b13_30_q_3_C4,b13_30_logvar_C3,b13_30_logvar_Cz,b13_30_logvar_C4
0,weibo2014,subject_01,session_01,0,right_hand_imagery,1.214693e-09,-8.251747e-09,-3.769403e-09,0.000002,0.000002,...,1.929559e-08,1.713261e-08,4.050321e-09,9.374276e-09,-1.674797e-10,4.685880e-08,4.067859e-08,-25.980474,-25.966007,-25.867661
1,weibo2014,subject_01,session_01,1,both_feet_imagery,-5.679104e-09,-1.393031e-08,6.417650e-09,0.000002,0.000002,...,1.884490e-09,9.481766e-08,8.473584e-08,6.182404e-08,-3.271543e-08,-5.221720e-08,-4.438596e-08,-25.796003,-25.690050,-25.858505
2,weibo2014,subject_01,session_01,2,both_feet_imagery,2.288404e-08,2.094546e-08,3.692862e-08,0.000003,0.000003,...,4.392534e-08,-3.335896e-08,-6.923555e-08,-6.923468e-08,6.994196e-08,7.042225e-08,7.446757e-08,-25.671808,-25.652632,-25.739395
3,weibo2014,subject_01,session_01,3,right_hand_imagery,-2.633061e-08,-3.243833e-08,-3.321287e-08,0.000002,0.000003,...,5.726635e-08,1.779817e-08,2.010461e-08,-5.468034e-09,-1.412661e-08,-8.202751e-09,1.772785e-08,-25.766415,-25.736692,-25.982655
4,weibo2014,subject_01,session_01,4,right_hand_imagery,1.180312e-09,9.214939e-09,-9.352116e-10,0.000002,0.000003,...,7.841227e-09,3.495800e-08,2.307013e-08,-1.813575e-08,-5.332830e-08,4.555810e-10,1.052103e-08,-25.599899,-25.656792,-25.647404


✅ Batch saved. Total rows written: 1200

Processing subjects 6–10
✅ Data loading complete.
✅ Filtering complete.


Subjects: 100%|██████████| 5/5 [00:02<00:00,  2.05it/s]


✅ Feature extraction complete: (1170, 95)
✅ Batch validation complete.
✅ Batch saved. Total rows written: 2370

Validating complete saved dataset...
✅ Complete dataset validation passed.
✅ Complete feature extraction finished.
✅ Electrode setup: setup_01
✅ Channels: ['C3', 'Cz', 'C4']
✅ Total rows: 2370
✅ Saved to: ../../../Datasets/Weibo2014/processed/Weibo2014_features_setup_01.csv
